# Identifying gene sets for inference

Builds 9 gene lists for inference, 3 criteria x 3 levels (high/mid/low) each. All nine now share
one recipe: rank a gene pool into three tiers by the criterion, then within each tier pick the top
15 CollecTRI TFs (by that tier's own score) plus 3-4 of each TF's curated targets, restricted to
the same tier -- so every list stays TFs-and-targets, not a plain ranked gene dump.

1. **Variability** -- rank the top-HVG pool by variability (dispersion), split into thirds; then
   apply the TF/target recipe within each third.
2. **Detection** -- rank genes by % of cells expressing them; 90-100/75-90/50-75th-percentile
   tiers (not equal thirds); then the TF/target recipe within each tier.
3. **Correlation** -- TFs and targets come from **CollecTRI** (curated TF-target regulons, not raw
   correlation); the "tier" here is a property of the (TF, target) *pair* (its |rho|) rather than of
   either gene alone -- High = the top 15 TFs whose curated targets correlate best with them on
   average, 3-4 curated targets each; Low sits just above the null co-expression cutoff (a
   permutation background, not an arbitrary number); Mid is in between.

TF/non-TF status and each TF's curated targets come from **CollecTRI**
(`resources/collectri_mouse.tsv`, [saezlab/CollecTRI](https://github.com/saezlab/CollecTRI),
pulled via the OmniPath REST API: 1,106 mouse TFs, 39,849 curated edges) -- used uniformly across
all three criteria, loaded once in the shared cell before Section 1. **AnimalTFDB 3.0**
(`resources/mouse_TF_AnimalTFDB3.txt`) is kept as a general TF/non-TF reference for use elsewhere
in the project -- built on Ensembl release 92 (GRCm38), matching the mouse assembly used to
annotate the inDrops data (Ensembl moved mouse to GRCm39 only at release 103, well after this data
was generated). Checked against `genes.txt`: 1,640/1,721 AnimalTFDB3 mouse TF symbols (95.3%) are
present, including every core hematopoietic TF (Gata1/2, Spi1, Fli1, Tal1, Cebpa, Klf1, Zfpm1,
Runx1, Ikzf1, ...).

All thresholds (N_HVG, the null-cutoff percentile, band widths, TF/target counts) are named
constants in one config cell below -- rerun downstream cells after changing them.


In [1]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.io as sio
import scipy.sparse as sp

HERE = os.path.dirname(os.path.abspath("__file__"))
SOURCE = "/home/gzu5140/TwINFER_KA/finalized_data/LARRY_data/filtered"
TF_LIST_PATH = os.path.join(HERE, "resources", "mouse_TF_AnimalTFDB3.txt")
COLLECTRI_PATH = os.path.join(HERE, "resources", "collectri_mouse.tsv")

# ---- config knobs (named, so retuning means editing one place then rerunning below) ----
MIN_DETECTION_FRAC = 0.05     # matches larry_raw_annotate.py's floor -- drop noise genes first
N_HVG = 2000                  # matches larry_raw_annotate.py's classifier-space HVG count
SEED = 0

N_PERMUTATIONS = 5            # independent cell-order shuffles per gene, for the null distribution
NULL_PERCENTILE = 95          # "average null cutoff for co-expression" = this percentile of |null r|

N_HIGH_TFS = 15                # top 15 TFs per correlation band (High/Mid/Low)
N_TARGETS_PER_TF = 4           # up to 4 curated targets per TF -- `.head(n)` below naturally
                                # falls back to 3 (or fewer) wherever a TF has fewer curated
                                # targets available in that band, so this reads as "3-4 each"
LOW_BAND_WIDTH = 1.15         # Low = pairs with |r| in [NULL_CUTOFF, NULL_CUTOFF * this]
MID_BAND = (0.35, 0.65)       # Mid = pairs whose |r| percentile (within the real, non-null TF-target
                               # distribution) falls in this range -- deliberately away from both the
                               # null floor and the top (High) tail

rng = np.random.default_rng(SEED)


/home/gzu5140/.conda/envs/twinfer-code/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## 0. Load data, filter, normalize

Same source as `larry_raw_annotate.py` (32,395 cells, `qc_filtered/`-style layout). The detection
floor is computed **per day** (worst/minimum detection fraction across days 2/4/6, not pooled
across all cells) -- matching `hvg_tfs_by_detection.py`'s `min_det = S[dets].min(axis=1)` convention:
a gene detected in 20% of day-2 cells but 0% of day-6 cells should not pass on the strength of an
average that a pooled calculation would report.


In [6]:
X = sio.mmread(os.path.join(SOURCE, "larry_qc_counts.mtx")).tocsr()
genes = pd.Index(open(os.path.join(SOURCE, "genes.txt")).read().split())
obs = pd.read_csv(os.path.join(SOURCE, "obs_metadata.csv"), index_col=0)
assert X.shape == (len(obs), len(genes))
print(f"loaded {X.shape[0]:,} cells x {X.shape[1]:,} genes")

day = obs["library"].astype(str).str.extract(r"_d(\d+)_", expand=False)
assert day.notna().all(), "could not parse day out of every library name"
day = day.astype(int).to_numpy()
days = sorted(set(day))
print(f"days present: {days}")

# per-day detection fraction, then the WORST (minimum) across days per gene -- not pooled across
# all cells (see markdown above)
per_day_frac = np.stack([
    np.asarray((X[day == d] > 0).sum(axis=0)).ravel() / (day == d).sum()
    for d in days
])
frac_expr = per_day_frac.min(axis=0)
gene_mask = frac_expr >= MIN_DETECTION_FRAC
print(f"per-day detection floor (min across {len(days)} days) >= {MIN_DETECTION_FRAC:.0%}: "
      f"{int(gene_mask.sum()):,} / {len(genes):,} genes")

genes = genes[gene_mask]
X = X[:, gene_mask]
X_day2 = X[day == 2]
obs_day2 = obs[day == 2]

frac_expr = frac_expr[gene_mask]

A = sc.AnnData(X=X.tocsr().astype(np.float32), obs=obs, var=pd.DataFrame(index=genes))
A.layers["counts"] = A.X.copy()
sc.pp.normalize_total(A, target_sum=1e4)
sc.pp.log1p(A)
print("normalized: log1p(CP10k)")

del X, obs, gene_mask, per_day_frac, X_day2, obs_day2
import gc; gc.collect()


loaded 32,395 cells x 25,289 genes
days present: [np.int64(2), np.int64(4), np.int64(6)]
per-day detection floor (min across 3 days) >= 5%: 5,136 / 25,289 genes
normalized: log1p(CP10k)


1475

## Shared: CollecTRI + the TF/target-subset recipe

Loaded here (before any of the three criteria) because all three now use it: CollecTRI decides
TF/non-TF status and gives each TF's curated targets, and `tf_target_subset()` is the shared
recipe -- within a given gene pool (a variability tier, a detection tier, or later a correlation
band), pick the top `N_HIGH_TFS` CollecTRI TFs ranked by that pool's own score, plus up to
`N_TARGETS_PER_TF` of each TF's curated targets that are **also in the same pool** -- so e.g. a
`variability_high` list stays made entirely of highly-variable genes, TFs and targets alike, never
a target pulled in from a lower tier just because its TF happens to be highly variable.


In [7]:
collectri_raw = pd.read_csv(COLLECTRI_PATH, sep="\t")

# _collectri.py's exact edge schema: sign (+1 activation/-1 repression/0 both), contested (no
# consensus direction), pmids (distinct PMID count), resources (distinct source-DB count).
def _edge_count(v):
    return len(set(str(v).split(";"))) if pd.notna(v) else 0

s, i = collectri_raw["is_stimulation"].astype(bool), collectri_raw["is_inhibition"].astype(bool)
collectri_raw["sign"] = np.where(s & i, 0, np.where(s, 1, -1))
collectri_raw["contested"] = (~(collectri_raw["consensus_stimulation"].astype(bool) |
                                 collectri_raw["consensus_inhibition"].astype(bool))).astype(int)
collectri_raw["effort"] = collectri_raw["curation_effort"].astype(int)
collectri_raw["pmids"] = collectri_raw["references"].apply(_edge_count)
collectri_raw["resources"] = collectri_raw["sources"].apply(_edge_count)
collectri = collectri_raw[collectri_raw["source_genesymbol"] != collectri_raw["target_genesymbol"]]  # self-pairs stripped
collectri_edge_meta = {
    (r.source_genesymbol, r.target_genesymbol): dict(sign=r.sign, contested=r.contested,
                                                      effort=r.effort, pmids=r.pmids, resources=r.resources)
    for r in collectri.itertuples(index=False)
}
collectri_tf_targets = collectri.groupby("source_genesymbol")["target_genesymbol"].apply(set).to_dict()
print(f"CollecTRI mouse: {collectri['source_genesymbol'].nunique():,} TFs, "
      f"{collectri['target_genesymbol'].nunique():,} distinct targets, {len(collectri):,} edges "
      f"(self-pairs stripped)")


def tf_target_subset(gene_pool, score, n_tfs=N_HIGH_TFS, n_targets=N_TARGETS_PER_TF):
    """From `gene_pool` (a band's full gene list), pick the top `n_tfs` CollecTRI TFs by `score`
    (a Series/dict giving that pool's own ranking value per gene), then for each TF add up to
    `n_targets` of its CollecTRI-curated targets that are ALSO in `gene_pool` (same-pool only).
    Falls back to fewer TFs/targets if the pool doesn't have enough. Returns (sorted gene list,
    detail list of (tf, [(target, score), ...]))."""
    pool_set = set(gene_pool)
    tfs_with_targets = []
    for tf in gene_pool:
        tgts = collectri_tf_targets.get(tf)
        if not tgts:
            continue
        in_pool = sorted((tgts & pool_set) - {tf})
        if in_pool:
            tfs_with_targets.append((tf, in_pool))
    tfs_with_targets.sort(key=lambda kv: score[kv[0]], reverse=True)
    chosen = tfs_with_targets[:n_tfs]

    genes_out, detail = set(), []
    for tf, in_pool in chosen:
        genes_out.add(tf)
        ranked = sorted(in_pool, key=lambda t: score[t], reverse=True)[:n_targets]
        genes_out.update(ranked)
        detail.append((tf, [(t, float(score[t])) for t in ranked]))
    return sorted(genes_out), detail


CollecTRI mouse: 1,106 TFs, 6,087 distinct targets, 39,847 edges (self-pairs stripped)


## 1. Variability -- high / mid / low

Rank the top-`N_HVG` pool by variability (scanpy's normalized dispersion), split into three equal
thirds by rank -- this is the tier ("Low" here means the *least* variable gene that still made the
HVG cut, not genome-wide least-variable; those are mostly detection-floor noise, already excluded
above). Within each tier, apply the same recipe as Correlation: the top `N_HIGH_TFS` CollecTRI TFs
(ranked by their own dispersion within the tier), plus up to `N_TARGETS_PER_TF` of each TF's
curated targets -- restricted to that same tier, so `variability_high` stays made entirely of
highly-variable genes.


In [8]:
sc.pp.highly_variable_genes(A, n_top_genes=N_HVG, batch_key="library")
hvg_genes = A.var_names[A.var["highly_variable"]]
assert len(hvg_genes) == N_HVG
hvg_rank = A.var.loc[hvg_genes, "dispersions_norm"].sort_values(ascending=False)

third = N_HVG // 3
variability_high_pool = hvg_rank.index[:third].tolist()
variability_mid_pool = hvg_rank.index[third:2 * third].tolist()
variability_low_pool = hvg_rank.index[2 * third:].tolist()

print(f"variability_high pool: {len(variability_high_pool)} genes, dispersion range "
      f"[{hvg_rank.iloc[third - 1]:.3f}, {hvg_rank.iloc[0]:.3f}]")
print(f"variability_mid  pool: {len(variability_mid_pool)} genes, dispersion range "
      f"[{hvg_rank.iloc[2 * third - 1]:.3f}, {hvg_rank.iloc[third]:.3f}]")
print(f"variability_low  pool: {len(variability_low_pool)} genes, dispersion range "
      f"[{hvg_rank.iloc[-1]:.3f}, {hvg_rank.iloc[2 * third]:.3f}]")

# same recipe as Correlation: top N_HIGH_TFS CollecTRI TFs within this tier (ranked by the tier's
# own dispersion), plus up to N_TARGETS_PER_TF of each TF's curated targets -- restricted to the
# SAME tier, so the final list stays entirely within its own variability band.
variability_high, variability_high_detail = tf_target_subset(variability_high_pool, hvg_rank)
variability_mid, variability_mid_detail = tf_target_subset(variability_mid_pool, hvg_rank)
variability_low, variability_low_detail = tf_target_subset(variability_low_pool, hvg_rank)

print(f"\nvariability_high: {len(variability_high)} genes "
      f"({len(variability_high_detail)} TFs + up to {N_TARGETS_PER_TF} same-tier curated targets each)")
print(f"variability_mid:  {len(variability_mid)} genes ({len(variability_mid_detail)} TFs)")
print(f"variability_low:  {len(variability_low)} genes ({len(variability_low_detail)} TFs)")
print("\nvariability_high TFs:", [tf for tf, _ in variability_high_detail])


variability_high pool: 666 genes, dispersion range [0.460, 11.316]
variability_mid  pool: 666 genes, dispersion range [0.177, 0.460]
variability_low  pool: 668 genes, dispersion range [-0.380, 0.177]

variability_high: 44 genes (15 TFs + up to 4 same-tier curated targets each)
variability_mid:  36 genes (15 TFs)
variability_low:  26 genes (9 TFs)

variability_high TFs: ['Pbx1', 'Gata2', 'Mycn', 'Id2', 'Sox4', 'Irf8', 'Jun', 'Bcl11a', 'Meis1', 'Myc', 'Nfkb2', 'Junb', 'Kdm5b', 'Litaf', 'Jund']


## 2. Detection -- high / mid / low

Per the established convention (`build_band_panel.py` / the `larry_detect_q*` panels): tiers are
**90-100% / 75-90% / 50-75%** detection percentile among genes surviving the floor -- not equal
thirds over the full 0-100% range. All three tiers sit in the upper half of the detection
distribution; nothing below the 50th percentile is used for this criterion. Within each tier,
same recipe as Variability/Correlation: top `N_HIGH_TFS` CollecTRI TFs (ranked by their own
detection fraction within the tier), plus up to `N_TARGETS_PER_TF` of each TF's curated targets,
restricted to that same tier.


In [9]:
detection = pd.Series(frac_expr, index=genes).sort_values(ascending=False)
p50, p75, p90 = np.percentile(detection.to_numpy(), [50, 75, 90])

detection_high_pool = detection[detection >= p90].index.tolist()
detection_mid_pool = detection[(detection >= p75) & (detection < p90)].index.tolist()
detection_low_pool = detection[(detection >= p50) & (detection < p75)].index.tolist()

print(f"detection thresholds: p50={p50:.1%}  p75={p75:.1%}  p90={p90:.1%}")
print(f"detection_high pool (>=90th pct): {len(detection_high_pool)} genes")
print(f"detection_mid  pool (75-90th pct): {len(detection_mid_pool)} genes")
print(f"detection_low  pool (50-75th pct): {len(detection_low_pool)} genes")

# same recipe as Variability/Correlation: top N_HIGH_TFS CollecTRI TFs within this tier (ranked by
# the tier's own detection fraction), plus up to N_TARGETS_PER_TF of each TF's curated targets --
# restricted to the SAME tier, so the final list stays entirely within its own detection band.
detection_high, detection_high_detail = tf_target_subset(detection_high_pool, detection)
detection_mid, detection_mid_detail = tf_target_subset(detection_mid_pool, detection)
detection_low, detection_low_detail = tf_target_subset(detection_low_pool, detection)

print(f"\ndetection_high: {len(detection_high)} genes "
      f"({len(detection_high_detail)} TFs + up to {N_TARGETS_PER_TF} same-tier curated targets each)")
print(f"detection_mid:  {len(detection_mid)} genes ({len(detection_mid_detail)} TFs)")
print(f"detection_low:  {len(detection_low)} genes ({len(detection_low_detail)} TFs)")
print("\ndetection_high TFs:", [tf for tf, _ in detection_high_detail])


detection thresholds: p50=12.1%  p75=21.4%  p90=37.6%
detection_high pool (>=90th pct): 514 genes
detection_mid  pool (75-90th pct): 770 genes
detection_low  pool (50-75th pct): 1285 genes

detection_high: 40 genes (14 TFs + up to 4 same-tier curated targets each)
detection_mid:  34 genes (14 TFs)
detection_low:  44 genes (15 TFs)

detection_high TFs: ['Ybx1', 'Npm1', 'Srsf2', 'Hdgf', 'Ybx3', 'Nono', 'Sfpq', 'Sf1', 'Srebf2', 'Nfkb1', 'Ctnnb1', 'Hmgb2', 'Trim28', 'Max']


## 3. Correlation -- high / mid / low

Rebuilt to match `helpers/_collectri.py` + `helpers/curated_vs_random_coexpr.py`'s exact method
(from `Transcriptomic Distance`, the closely related project on this same LARRY dataset), not a
from-scratch design. (CollecTRI itself was already loaded in the shared cell before Section 1 --
here it's scoped down to the HVG pool.)

- **CollecTRI edge schema** = `_collectri.py`'s fields exactly (`sign`, `contested`, `effort`,
  `pmids`, `resources`), self-pairs stripped. Candidate TFs = CollecTRI source genes present in the
  HVG pool; each TF's candidate targets are restricted to *its own* curated targets (intersected
  with HVG) -- not the whole non-TF HVG pool.
- **Spearman**, not Pearson (rank-transform then Pearson on ranks), matching
  `curated_vs_random_coexpr.py`.
- **Null = detection-decile-matched decoys**, not a per-gene shuffle: for each curated (TF, target)
  edge, the decoy is the same TF against a random target of the same detection decile -- so the only
  difference from a curated edge is whether CollecTRI records it, exactly their design. `NULL_CUTOFF`
  is the `NULL_PERCENTILE`th percentile of this decoy |rho| distribution.
- **Mann-Whitney U / AUC** reported (curated |rho| vs. matched decoys), same diagnostic they used.
  Their own result on LARRY (`curated_vs_random_coexpr.py`, log1pPF, days 2&4 only): AUC=0.52,
  curated median |rho|=0.036 vs. decoy median=0.035 -- curated edges carry almost no extra
  co-expression signal here. Expect our own numbers (log1pCP10k, all days, HVG-restricted) to land
  in the same weak regime -- "High" should be read as *relatively* highest, not as strong signal in
  absolute terms.

High/Mid/Low are still built the same way as Variability/Detection: top `N_HIGH_TFS` TFs, up to
`N_TARGETS_PER_TF` targets each -- but banded by |rho| relative to `NULL_CUTOFF` instead of by a
per-gene score, since a correlation band is a property of the (TF, target) *pair*, not of either
gene alone.


In [10]:
# collectri_tf_targets was already loaded in the shared "CollecTRI + TF/target-subset recipe"
# cell above (before Variability) -- Correlation just reuses it, now scoped to the HVG pool.
hvg_set = set(hvg_genes)
candidate_tfs = [g for g in hvg_genes if g in collectri_tf_targets]
tf_curated_targets = {
    tf: sorted((collectri_tf_targets[tf] & hvg_set) - {tf})
    for tf in candidate_tfs
}
candidate_tfs = [tf for tf in candidate_tfs if len(tf_curated_targets[tf]) > 0]
print(f"candidate TFs (CollecTRI ∩ HVG, with >=1 curated target also in HVG): {len(candidate_tfs)}")

Xh = A[:, hvg_genes].X
Xh = Xh.toarray() if sp.issparse(Xh) else np.asarray(Xh)
hvg_index = {g: i for i, g in enumerate(hvg_genes)}
tf_idx = np.array([hvg_index[g] for g in candidate_tfs])
# Decoy/target universe is the FULL HVG pool, matching curated_vs_random_coexpr.py's use of the
# whole detectable gene set (their `gl`/`det_all[keep]`) -- NOT restricted to genes that happen to
# already be a curated target for someone. A decoy drawn only from other curated targets would be
# a biased comparison; the whole point of their design is "any gene of matched detection".
candidate_targets = list(hvg_genes)
tgt_idx = np.arange(len(hvg_genes))

target_index = hvg_index  # candidate_targets == hvg_genes, so this is just an alias
curated_mask = np.zeros((len(candidate_tfs), len(candidate_targets)), dtype=bool)
for i, tf in enumerate(candidate_tfs):
    for tgt in tf_curated_targets[tf]:
        curated_mask[i, target_index[tgt]] = True
print(f"curated (TF,target) cells to rank: {curated_mask.sum():,} / {curated_mask.size:,} "
      f"dense-matrix entries")


def spearman_matrix(mat, rows_idx, cols_idx):
    """Spearman correlation: Pearson on rank-transformed columns, matching
    curated_vs_random_coexpr.py's Rk = rankdata(X); Rk -= mean; Rk /= std; rho = Rk.T @ Rk / n."""
    from scipy.stats import rankdata
    Rk = np.apply_along_axis(rankdata, 0, mat).astype(np.float64)
    Rk = (Rk - Rk.mean(axis=0)) / np.maximum(Rk.std(axis=0), 1e-9)
    Rr, Rc = Rk[:, rows_idx], Rk[:, cols_idx]
    return (Rr.T @ Rc) / Rr.shape[0]


R = spearman_matrix(Xh, tf_idx, tgt_idx)  # (n_candidate_tfs, n_hvg_genes), Spearman rho
print(f"real TF x (full HVG pool) Spearman matrix: {R.shape}, "
      f"|rho| range over curated cells [{np.abs(R)[curated_mask].min():.3f}, "
      f"{np.abs(R)[curated_mask].max():.3f}]")


candidate TFs (CollecTRI ∩ HVG, with >=1 curated target also in HVG): 97
curated (TF,target) cells to rank: 1,129 / 194,000 dense-matrix entries
real TF x (full HVG pool) Spearman matrix: (97, 2000), |rho| range over curated cells [0.000, 0.353]


In [11]:
from scipy import stats

# curated_vs_random_coexpr.py's exact decoy design: for each curated (TF, target) edge, draw a
# decoy = the SAME TF against a RANDOM gene of the full HVG pool matched on detection DECILE (not a
# per-gene shuffle, and not restricted to other curated targets) -- so the only difference from a
# curated edge is whether CollecTRI records it, not expression level or curation status of the decoy.
det_of = pd.Series(frac_expr, index=genes)
hvg_det = det_of.loc[hvg_genes].to_numpy()
decile_edges = np.quantile(hvg_det, np.linspace(0, 1, 11)[1:-1])
decile = np.digitize(hvg_det, decile_edges)
by_decile = {d: np.flatnonzero(decile == d) for d in range(10)}
print("decile pool sizes:", {d: len(v) for d, v in by_decile.items()})

real_pairs = {(i, j) for i in range(len(candidate_tfs)) for j in range(len(candidate_targets)) if curated_mask[i, j]}
obs_absr, dec_absr = [], []
for (i, j) in real_pairs:
    obs_absr.append(abs(R[i, j]))
    pool = by_decile[decile[j]]
    for _ in range(20):
        c = int(rng.choice(pool))
        if c != j and (i, c) not in real_pairs:
            dec_absr.append(abs(R[i, c]))
            break
obs_absr, dec_absr = np.array(obs_absr), np.array(dec_absr)
print(f"curated pairs: {len(obs_absr)}  |  matched decoys found: {len(dec_absr)}")

u = stats.mannwhitneyu(obs_absr, dec_absr, alternative="greater")
auc = u.statistic / (len(obs_absr) * len(dec_absr))
print(f"curated |rho|: median {np.median(obs_absr):.4f}  mean {obs_absr.mean():.4f}")
print(f"decoy   |rho|: median {np.median(dec_absr):.4f}  mean {dec_absr.mean():.4f}  "
      f"(detection-decile-matched, drawn from the full HVG pool)")
print(f"curated vs matched decoys: AUC {auc:.4f}  P = {u.pvalue:.3e}  "
      f"(AUC 0.5 = a curated edge is no more co-expressed than a matched non-edge)")

NULL_CUTOFF = float(np.percentile(dec_absr, NULL_PERCENTILE))
print(f"\nNULL_CUTOFF ({NULL_PERCENTILE}th percentile of the matched-decoy |rho|) = {NULL_CUTOFF:.4f}")
real_above_null = (obs_absr > NULL_CUTOFF).mean()
print(f"fraction of REAL, CURATED TF-target pairs clearing NULL_CUTOFF: {real_above_null:.1%}")


decile pool sizes: {0: 199, 1: 201, 2: 200, 3: 198, 4: 202, 5: 197, 6: 203, 7: 200, 8: 200, 9: 200}
curated pairs: 1129  |  matched decoys found: 1129
curated |rho|: median 0.0360  mean 0.0494
decoy   |rho|: median 0.0324  mean 0.0423  (detection-decile-matched, drawn from the full HVG pool)
curated vs matched decoys: AUC 0.5490  P = 2.767e-05  (AUC 0.5 = a curated edge is no more co-expressed than a matched non-edge)

NULL_CUTOFF (95th percentile of the matched-decoy |rho|) = 0.1071
fraction of REAL, CURATED TF-target pairs clearing NULL_CUTOFF: 9.8%


In [12]:
# ---- High: TFs with the best average |r| against their OWN curated targets, plus those top targets ----
R_masked = np.where(curated_mask, np.abs(R), np.nan)

tf_mean_absr = pd.Series(np.nanmean(R_masked, axis=1), index=candidate_tfs).sort_values(ascending=False)
high_tfs = tf_mean_absr.index[:N_HIGH_TFS].tolist()
print("top TFs by mean |r| with their own CollecTRI-curated targets:")
print(tf_mean_absr.head(N_HIGH_TFS).round(3))

correlation_high = set(high_tfs)
high_detail = []
for tf in high_tfs:
    i = candidate_tfs.index(tf)
    row = pd.Series(R_masked[i], index=candidate_targets).dropna().sort_values(ascending=False)
    top_targets = row.index[:N_TARGETS_PER_TF].tolist()
    correlation_high.update(top_targets)
    high_detail.append((tf, list(zip(top_targets, row.iloc[:N_TARGETS_PER_TF].round(3).tolist()))))
correlation_high = sorted(correlation_high)
print(f"\ncorrelation_high: {len(correlation_high)} genes "
      f"({len(high_tfs)} TFs + up to {N_TARGETS_PER_TF} curated targets each)")

# ---- Low / Mid: CURATED pairs only, banded by |r|, same TF/target shape as High for comparability ----
all_pairs = pd.DataFrame(
    [(candidate_tfs[i], candidate_targets[j], abs(R[i, j]))
     for i in range(len(candidate_tfs)) for j in range(len(candidate_targets)) if curated_mask[i, j]],
    columns=["tf", "target", "abs_r"])

low_hi = NULL_CUTOFF * LOW_BAND_WIDTH
mid_lo, mid_hi = np.percentile(all_pairs["abs_r"], [MID_BAND[0] * 100, MID_BAND[1] * 100])
print(f"\nNULL_CUTOFF={NULL_CUTOFF:.3f}  low band=[{NULL_CUTOFF:.3f}, {low_hi:.3f}]  "
      f"mid band=[{mid_lo:.3f}, {mid_hi:.3f}]")

low_pairs = all_pairs[all_pairs["abs_r"].between(NULL_CUTOFF, low_hi)]
mid_pairs = all_pairs[all_pairs["abs_r"].between(mid_lo, mid_hi)]
print(f"curated pairs in low band: {len(low_pairs):,}  |  curated pairs in mid band: {len(mid_pairs):,}")


def pick_list(pairs_in_band, n_tfs=N_HIGH_TFS, n_targets=N_TARGETS_PER_TF):
    tf_counts = pairs_in_band["tf"].value_counts()
    chosen_tfs = tf_counts.index[:n_tfs].tolist()
    genes_out, detail = set(chosen_tfs), []
    for tf in chosen_tfs:
        sub = pairs_in_band[pairs_in_band["tf"] == tf].sort_values("abs_r", ascending=False)
        tgts = sub["target"].head(n_targets).tolist()
        genes_out.update(tgts)
        detail.append((tf, list(zip(tgts, sub["abs_r"].head(n_targets).round(3).tolist()))))
    return sorted(genes_out), detail


correlation_low, low_detail = pick_list(low_pairs)
correlation_mid, mid_detail = pick_list(mid_pairs)
print(f"\ncorrelation_low: {len(correlation_low)} genes")
print(f"correlation_mid: {len(correlation_mid)} genes")


top TFs by mean |r| with their own CollecTRI-curated targets:
Bcl11a    0.142
Cebpd     0.117
Cebpb     0.095
Patz1     0.091
Gata2     0.083
Jun       0.077
Npm1      0.076
Spi1      0.074
Id2       0.071
Tfec      0.069
Irf8      0.068
Kdm2b     0.066
Myc       0.065
Lmo2      0.063
Erg       0.060
dtype: float64

correlation_high: 44 genes (15 TFs + up to 4 curated targets each)

NULL_CUTOFF=0.107  low band=[0.107, 0.123]  mid band=[0.028, 0.049]
curated pairs in low band: 33  |  curated pairs in mid band: 339

correlation_low: 35 genes
correlation_mid: 60 genes


## 4. Summary -- all 9 lists, saved


In [ ]:
import json

gene_sets = {
    "variability_high": variability_high,
    "variability_mid": variability_mid,
    "variability_low": variability_low,
    "detection_high": detection_high,
    "detection_mid": detection_mid,
    "detection_low": detection_low,
    "correlation_high": correlation_high,
    "correlation_mid": correlation_mid,
    "correlation_low": correlation_low,
}

for name, genes_ in gene_sets.items():
    print(f"{name:<18} {len(genes_):>5} genes")

out_path = os.path.join(HERE, "resources", "gene_sets.json")
json.dump(gene_sets, open(out_path, "w"), indent=2)
print(f"\nwrote {out_path}")

# per-TF detail (which TF pulled in which targets, at what score) for all 9 lists, for inspection
gene_sets_detail = {
    "variability": {"high": variability_high_detail, "mid": variability_mid_detail,
                     "low": variability_low_detail},
    "detection": {"high": detection_high_detail, "mid": detection_mid_detail,
                   "low": detection_low_detail},
    "correlation": {"high": high_detail, "mid": mid_detail, "low": low_detail},
}
detail_path = os.path.join(HERE, "resources", "gene_sets_detail.json")
json.dump(gene_sets_detail, open(detail_path, "w"), indent=2)
print(f"wrote {detail_path}")


variability_high      44 genes
variability_mid       36 genes
variability_low       26 genes
detection_high        40 genes
detection_mid         34 genes
detection_low         44 genes
correlation_high      44 genes
correlation_mid       60 genes
correlation_low       35 genes

wrote /gpfs/projects/b1255/hzhang/TwINFER_KA/code/TwINFER/paper_analysis/larry_hematopoiesis_validation/resources/gene_sets.json
wrote /gpfs/projects/b1255/hzhang/TwINFER_KA/code/TwINFER/paper_analysis/larry_hematopoiesis_validation/resources/gene_sets_detail.json


: 